# Jam 3-6: Model Prediksi Kemacetan (Random Forest)

**v2 — update setelah data lengkap (weekday + weekend, 6 segmen penuh).**

Perubahan dari versi sebelumnya:
1. `is_peak` sekarang diturunkan dari `hour` saja (independen dari target) — versi lama bocor dari `congestion_level`.
2. Target training memakai `congestion_level` yang sudah dilabeli di sumber data (HIGH/MEDIUM/LOW), bukan dihitung ulang dari threshold multiplier sembarangan — supaya konsisten dengan label yang sudah divalidasi periset.
3. Ekspansi jam per window sekarang meng-handle window yang wrap tengah malam (mis. 19:00-06:00) dan window 24 jam penuh (weekend), tanpa overlap/duplikat di jam batas.
4. Tambah mapping `level -> multiplier` representatif untuk dipakai endpoint `/route`.
5. Tambah fungsi hitung waktu tempuh & emisi sesuai formula di `research_sources` (T_actual, Carbon Emission).

In [1]:
!pip install supabase pandas scikit-learn python-dotenv joblib

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.8 MB 10.2 MB/s eta 0:00:01
   ---------------- ----------------------- 3.9/9.8 MB 11.0 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.8 MB 11.1 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.8 MB 10.6 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 10.4 MB/s  0:00:01

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- -----------

In [2]:
import os
import pandas as pd
import numpy as np
from supabase import create_client, Client
from dotenv import load_dotenv
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

load_dotenv(dotenv_path='../.env')

url: str = os.environ.get("SUPABASE_URL")
key: str = os.environ.get("SUPABASE_ANON_KEY")
supabase: Client = create_client(url, key)

### 1. Fetch Data dari Supabase

Asumsi: tabel `congestion_multipliers` di Supabase sudah versi bersih (kolom `segment_id`, `day_type`, `hour_window`, `hour_start`, `hour_end`, `multiplier`, `congestion_level`, `description` saja — tanpa kolom catatan metodologi tambahan dari spreadsheet mentah).

In [3]:
response_segments = supabase.table("road_segments").select("*").execute()
response_multipliers = supabase.table("congestion_multipliers").select("*").execute()

df_segments = pd.DataFrame(response_segments.data)
df_multipliers = pd.DataFrame(response_multipliers.data)

print(f"road_segments: {len(df_segments)} baris")
print(f"congestion_multipliers: {len(df_multipliers)} baris")
df_multipliers.head(10)

road_segments: 6 baris
congestion_multipliers: 25 baris


,id,segment_id,day_type,hour_window,hour_start,hour_end,multiplier,congestion_level,description,created_at
0,7846685e-a122-444d-91fb-fb5e1918e15e,SEG001,weekday,06:00-09:00,6,9,1.85,HIGH,Morning peak: Container truck and factory work...,2026-08-15T13:34:21.370261+00:00
1,fde8612e-99e1-4f9a-b435-1dbb48fe2a3b,SEG001,weekday,09:00-15:00,9,15,1.20,MEDIUM,Normal logistics operating hours,2026-08-15T13:34:21.370261+00:00
2,3d3e2f8f-8e9a-405a-b281-38ada1dc933c,SEG001,weekday,15:00-19:00,15,19,2.10,HIGH,Evening peak & Batu Ampar cargo vessel cut-off...,2026-08-15T13:34:21.370261+00:00
3,f1362f5f-9b68-447b-a993-e864fcb66f86,SEG001,weekday,19:00-06:00,19,6,1.00,LOW,Free-flowing traffic (off-peak night hours),2026-08-15T13:34:21.370261+00:00
4,398fe496-2803-4280-9e19-fac616f1abca,SEG001,weekend,00:00-24:00,0,24,1.10,LOW,"Weekend: Minimal factory activity, corridor ru...",2026-08-15T13:34:21.370261+00:00
5,d805453c-35d8-4103-a35e-18c7e42dfa29,SEG002,weekday,06:30-08:30,6,8,1.50,HIGH,Heavy private vehicle and public transport con...,2026-08-15T13:34:21.370261+00:00
6,77287190-42f7-4199-a3fd-834e589fbd60,SEG002,weekday,08:30-16:30,8,16,1.20,MEDIUM,Moderate urban traffic flow,2026-08-15T13:34:21.370261+00:00
7,4752193f-4172-412e-9632-05e749881179,SEG002,weekday,16:30-18:30,16,18,1.60,HIGH,Evening rush hour congestion,2026-08-15T13:34:21.370261+00:00
8,cb9e89a9-e422-427f-875f-9b618094840d,SEG002,weekday,18:30-06:30,18,6,1.00,LOW,Free-flowing traffic (night and early morning),2026-08-15T13:34:21.370261+00:00
9,cdf198ae-cbf6-46fa-932d-502926881132,SEG002,weekend,00:00-24:00,0,24,1.25,LOW,Weekend: Residents commuting to shopping centers,2026-08-15T13:34:21.370261+00:00


### 2. Sanity Check Coverage

Cek cepat: tiap segmen harus punya baris untuk `weekday` dan `weekend`, dan gabungan window jamnya harus menutup 24 jam penuh tanpa celah. Kalau ada segmen yang belum lengkap, notebook akan tetap jalan tapi modelnya tidak akan bisa memprediksi kombinasi yang tidak ada datanya.

In [4]:
coverage = df_multipliers.groupby(['segment_id', 'day_type']).size().unstack(fill_value=0)
print("Jumlah window per segmen x day_type (0 = tidak ada data):")
coverage

Jumlah window per segmen x day_type (0 = tidak ada data):


day_type,weekday,weekend
segment_id,,
SEG001,4,1
SEG002,4,1
SEG003,4,1
SEG004,3,1
SEG005,1,2
SEG006,2,1


### 3. Generate Synthetic Data

Tiap baris di `congestion_multipliers` di-expand jadi N baris sintetis per jam dalam window-nya, dengan noise kecil di sekitar multiplier dasar. Target (`congestion_level`) diambil langsung dari label yang sudah ada di sumber data — bukan dihitung ulang dari threshold — supaya konsisten dengan hasil kalibrasi periset (mis. SEG001 09:00-15:00 punya multiplier 1.2 tapi dilabeli MEDIUM, bukan LOW).

In [5]:
def expand_hours(hour_start, hour_end):
    """Ekspansi window jam [start, end) jadi list jam individual.
    Handle wrap tengah malam (mis. 19 -> 6) dan window 24 jam penuh (0 -> 24)."""
    start = int(hour_start)
    end = int(hour_end)
    if end == 24:
        end = 0
    if end == start:
        # window 24 jam penuh (mis. weekend 00:00-24:00)
        return list(range(0, 24))
    if end < start:
        # wrap tengah malam, mis. 19 -> 24 lalu 0 -> 6
        return list(range(start, 24)) + list(range(0, end))
    return list(range(start, end))


def is_peak_hour(hour):
    """Diturunkan murni dari jam, independen dari target/level."""
    return 1 if (6 <= hour <= 9) or (16 <= hour <= 19) else 0


SAMPLES_PER_HOUR = 50
NOISE_RANGE = 0.10  # +/- 10%

synthetic_data = []
np.random.seed(42)

for _, row in df_multipliers.iterrows():
    segment_id = row['segment_id']
    day_type = row['day_type']
    base_multiplier = float(row['multiplier'])
    level = str(row['congestion_level']).lower()  # ground truth dari sumber data

    corridor_match = df_segments[df_segments['segment_id'] == segment_id]['corridor_type']
    if corridor_match.empty:
        print(f"WARNING: segment_id {segment_id} tidak ditemukan di road_segments, dilewati")
        continue
    corridor = corridor_match.values[0]

    for hour in expand_hours(row['hour_start'], row['hour_end']):
        for _ in range(SAMPLES_PER_HOUR):
            noise = np.random.uniform(-NOISE_RANGE, NOISE_RANGE)
            multiplier = base_multiplier * (1 + noise)

            synthetic_data.append({
                'segment_id': segment_id,
                'corridor_type': corridor,
                'day_type': day_type,
                'hour': hour,
                'is_peak': is_peak_hour(hour),
                'multiplier': multiplier,
                'congestion_level': level,
            })

df_train = pd.DataFrame(synthetic_data)
print(f"Total synthetic data generated: {len(df_train)}")
df_train['congestion_level'].value_counts()

Total synthetic data generated: 14400


congestion_level
low       11750
medium     1750
high        900
Name: count, dtype: int64

### 4. Feature Engineering & One-Hot Encoding

In [6]:
features = df_train[['hour', 'day_type', 'corridor_type', 'is_peak']]
target = df_train['congestion_level']

X = pd.get_dummies(features, columns=['day_type', 'corridor_type'])
y = target

feature_columns = list(X.columns)
print("Fitur input:", feature_columns)

# Sanity check: pastikan day_type_weekend & day_type_weekday dua-duanya ada
missing_daytype_cols = [c for c in ['day_type_weekday', 'day_type_weekend'] if c not in feature_columns]
if missing_daytype_cols:
    print(f"WARNING: kolom berikut tidak muncul di training data: {missing_daytype_cols} "
          f"-> prediksi untuk kategori ini akan gagal di API karena feature mismatch")
else:
    print("OK: day_type_weekday dan day_type_weekend dua-duanya ada di training data")

Fitur input: ['hour', 'is_peak', 'day_type_weekday', 'day_type_weekend', 'corridor_type_access_road', 'corridor_type_industrial_arterial', 'corridor_type_urban_arterial']
OK: day_type_weekday dan day_type_weekend dua-duanya ada di training data


### 5. Train Random Forest Model

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.951
              precision    recall  f1-score   support

        high       0.72      0.82      0.77       180
         low       0.97      0.99      0.98      2350
      medium       0.94      0.75      0.84       350

    accuracy                           0.95      2880
   macro avg       0.88      0.85      0.86      2880
weighted avg       0.95      0.95      0.95      2880



### 6. Mapping Level -> Multiplier Representatif

Dipakai endpoint `/route` untuk menghitung waktu tempuh terbobotinya, tanpa perlu model regresi terpisah — cukup ambil rata-rata multiplier training per kelas.

In [8]:
level_to_multiplier = df_train.groupby('congestion_level')['multiplier'].mean().round(3).to_dict()
print("Mapping level -> multiplier representatif:")
level_to_multiplier

Mapping level -> multiplier representatif:


{'high': 1.818, 'low': 1.066, 'medium': 1.227}

### 7. Fungsi Waktu Tempuh & Emisi

Sesuai formula di `research_sources`:
- `T_actual = (Distance / Free_Flow_Speed) * Multiplier * 60` menit
- `Fuel = (Distance * 0.25 L/km) + (Idle_Hours * 2.25 L/hr)`
- `Emission (kg CO2) = Fuel * 2.68`

Idle hours dihitung dari selisih waktu tempuh aktual dengan waktu tempuh free-flow (tanpa macet).

In [9]:
FUEL_PER_KM = 0.25       # L/km, konsumsi normal berjalan
FUEL_PER_IDLE_HOUR = 2.25  # L/jam, konsumsi saat idling/stop-and-go
CO2_PER_LITER = 2.68      # kg CO2 per liter diesel (IPCC 2006)


def calculate_travel_time_minutes(distance_km, free_flow_speed_kmh, multiplier):
    free_flow_minutes = (distance_km / free_flow_speed_kmh) * 60
    return free_flow_minutes * multiplier


def calculate_emission_kg(distance_km, free_flow_speed_kmh, multiplier):
    free_flow_minutes = (distance_km / free_flow_speed_kmh) * 60
    actual_minutes = free_flow_minutes * multiplier
    idle_hours = max(0, (actual_minutes - free_flow_minutes)) / 60
    fuel_liters = (distance_km * FUEL_PER_KM) + (idle_hours * FUEL_PER_IDLE_HOUR)
    return round(fuel_liters * CO2_PER_LITER, 2)


# Contoh pemakaian: SEG001, 12.5 km, free-flow 40 km/jam, multiplier peak sore 2.1x
example_time = calculate_travel_time_minutes(12.5, 40.0, 2.1)
example_emission = calculate_emission_kg(12.5, 40.0, 2.1)
print(f"Contoh: waktu tempuh {example_time:.1f} menit, emisi {example_emission} kg CO2")

Contoh: waktu tempuh 39.4 menit, emisi 10.45 kg CO2


### 8. Uji Prediksi Manual (Sanity Check)

Cek beberapa kombinasi known-value sebelum diintegrasikan ke endpoint `/congestion`, termasuk kombinasi weekend yang sebelumnya tidak ada datanya sama sekali.

In [10]:
def predict_congestion(segment_id, day_type, hour):
    corridor = df_segments[df_segments['segment_id'] == segment_id]['corridor_type'].values[0]
    row = pd.DataFrame([{
        'hour': hour,
        'is_peak': is_peak_hour(hour),
        'day_type': day_type,
        'corridor_type': corridor,
    }])
    row_encoded = pd.get_dummies(row, columns=['day_type', 'corridor_type'])
    row_encoded = row_encoded.reindex(columns=feature_columns, fill_value=0)
    level = rf_model.predict(row_encoded)[0]
    return level, level_to_multiplier[level]


test_cases = [
    ('SEG001', 'weekday', 7),   # ekspektasi: peak pagi, high
    ('SEG001', 'weekday', 11),  # ekspektasi: jam operasional, medium
    ('SEG001', 'weekend', 12),  # ekspektasi: weekend, low
    ('SEG005', 'weekend', 10),  # ekspektasi: weekend wisata Barelang, medium
]

for segment_id, day_type, hour in test_cases:
    level, mult = predict_congestion(segment_id, day_type, hour)
    print(f"{segment_id} | {day_type} | jam {hour}: {level} (multiplier ~{mult})")

SEG001 | weekday | jam 7: high (multiplier ~1.818)
SEG001 | weekday | jam 11: low (multiplier ~1.066)
SEG001 | weekend | jam 12: low (multiplier ~1.066)
SEG005 | weekend | jam 10: medium (multiplier ~1.227)


### 9. Export Model (.joblib)

Package model beserta `feature_columns` (untuk reindex fitur saat inferensi) dan `level_to_multiplier` (untuk endpoint `/route`) agar backend FastAPI tidak perlu mengulang logic training.

In [11]:
model_package = {
    'model': rf_model,
    'feature_columns': feature_columns,
    'level_to_multiplier': level_to_multiplier,
}

joblib.dump(model_package, '../rf_model.joblib')
print("Model berhasil diexport ke backend/rf_model.joblib!")

Model berhasil diexport ke backend/rf_model.joblib!
